# Mohammad Amin Kiani - 4043644008
# NLP - HW3 : Recurrent To Attention
# ui.ac.ir 404-405

1

In [1]:
# # نصب کتابخانه‌های مورد نیاز برای پردازش مدل‌های زبانی، کوانتیزه‌سازی و تنظیم دقیق
!pip install -q -U transformers accelerate bitsandbytes peft datasets trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.2 MB/s eta 0:00:00


2

In [1]:
# نصب نسخه‌های پایدار و هماهنگ کتابخانه‌ها
!pip install -q -U "transformers>=4.40.0" "accelerate>=0.28.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" datasets sklearn

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


3

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" "transformers>=4.40.0" accelerate peft datasets scikit-learn tqdm

#### 3:

In [ ]:
import torch
import time
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import PrefixTuningConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# ۱. بارگذاری دیتاست و توکنایزر
print("Loading Dataset and Tokenizer...")
dataset = load_dataset("hezarai/sentiment-dksf")
train_data = dataset['train']
test_data = dataset['test']

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
label_map = {0: "منفی", 1: "خنثی", 2: "مثبت"}

# تابع پردازش داده‌ها
def preprocess_function(example):
    prompt = f"متن: {example['text']}\nاحساس:"
    label_text = label_map.get(example['label'], "")
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": label_text}
    ]
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(formatted_text, truncation=True, max_length=512)

print("Tokenizing data...")
tokenized_train_data = train_data.map(preprocess_function, remove_columns=train_data.column_names)

# ۲. بارگذاری مدل پایه و تنظیمات Prefix Tuning
print("\nLoading Model for Prefix Tuning...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_pt = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

# غیرفعال کردن Gradient Checkpointing جهت سازگاری با Prefix Tuning
model_pt = prepare_model_for_kbit_training(model_pt, use_gradient_checkpointing=False)

prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20
)

model_pt = get_peft_model(model_pt, prefix_config)

print("\nPrefix Tuning Trainable Parameters:")
model_pt.print_trainable_parameters()

# ۳. تنظیمات آموزش (Training Arguments)
training_args_pt = TrainingArguments(
    output_dir="./prefix_sentiment",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer_pt = Trainer(
    model=model_pt,
    train_dataset=tokenized_train_data,
    args=training_args_pt,
    data_collator=data_collator
)

# ۴. شروع آموزش و محاسبه زمان
print("\nStarting Prefix Tuning Training...")
start_pt_train = time.time()

trainer_pt.train()

end_pt_train = time.time()
pt_training_time = end_pt_train - start_pt_train
print(f"\n==========================================")
print(f"Prefix Tuning Training Time: {pt_training_time:.2f} seconds ({pt_training_time/60:.2f} minutes)")
print(f"==========================================")

# ۵. ذخیره آداپتور Prefix Tuning
trainer_pt.model.save_pretrained("adapter_prefix")
print("Adapter saved successfully in 'adapter_prefix' folder.")


# ==========================================
# ۶. ارزیابی مدل Prefix Tuning روی داده‌های تست
# ==========================================

def clean_and_extract_label(gen_text):
    gen_text = gen_text.strip()
    if "مثبت" in gen_text or gen_text.startswith("مث"):
        return "مثبت"
    elif "منفی" in gen_text or gen_text.startswith("من"):
        return "منفی"
    elif "خنثی" in gen_text or gen_text.startswith("خ"):
        return "خنثی"
    return gen_text

def evaluate_prefix_model(model, tokenizer, test_dataset, num_samples=500):
    test_subset = test_dataset.select(range(min(num_samples, len(test_dataset))))

    true_labels = []
    pred_labels = []
    errors = []

    model.eval()
    for row in tqdm(test_subset):
        prompt = f"متن: {row['text']}\nاحساس:"
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        gen_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = clean_and_extract_label(gen_text)
        true_label = label_map.get(row['label'], "")

        true_labels.append(true_label)
        pred_labels.append(pred)

        if pred != true_label and len(errors) < 5:
            errors.append({"text": row['text'], "true": true_label, "pred": pred, "raw": gen_text})

    acc = accuracy_score(true_labels, pred_labels)
    labels_list = ["مثبت", "منفی", "خنثی"]
    f_macro = f1_score(true_labels, pred_labels, labels=labels_list, average='macro', zero_division=0)

    return acc, f_macro, errors

print("\nEvaluating Prefix Tuning Trained Model...")
acc_pt, f_macro_pt, error_samples_pt = evaluate_prefix_model(trainer_pt.model, tokenizer, test_data)

print(f"\n==========================================")
print(f"Prefix Tuning Accuracy : {acc_pt:.4f} ({acc_pt*100:.2f}%)")
print(f"Prefix Tuning F-Macro  : {f_macro_pt:.4f}")
print(f"==========================================")

print("\n--- Error Samples (Prefix Tuning) ---")
for e in error_samples_pt:
    print(f"Text: {e['text']}\nTrue: {e['true']} | Predicted: {e['pred']} (Raw: '{e['raw']}')\n-")

Loading Dataset and Tokenizer...


README.md:   0%|          | 0.00/705 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.44MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  266kB            

...


Prefix Tuning Trainable Parameters:
trainable params: 122,880 || all params: 494,155,648 || trainable%: 0.0249

Starting Prefix Tuning Training...


Step,Training Loss
50,12.735897
100,10.815051
150,9.058045
200,7.699527
250,6.678441
300,5.906814
350,5.501271
400,5.020912
450,4.753682
500,4.626589



Prefix Tuning Training Time: 2370.23 seconds (39.50 minutes)
Adapter saved successfully in 'adapter_prefix' folder.

Evaluating Prefix Tuning Trained Model...


100%|██████████| 500/500 [02:22<00:00,  3.50it/s]


Prefix Tuning Accuracy : 0.3080 (30.80%)
Prefix Tuning F-Macro  : 0.2609

--- Error Samples (Prefix Tuning) ---
Text: عطره و بیشتر به درد لباس میخوره، چون اصلا پخش بو نداره و ظرف ۴-۵ روز تبخیر میشه، بوی خوبی داره به شرطی که جلوی بینی بگیرید
True: منفی | Predicted: فیسای ک (Raw: 'فیسای ک')
-
Text: اصلا بدرد نمیخوره.بعد از دو سه روز کنده میشه.زود کثیف میشه و کثیفی رو بخودش میگیره.
True: منفی | Predicted: احساس گون (Raw: 'احساس گون')
-
Text: قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا
True: خنثی | Predicted: احساس: 2 (Raw: 'احساس: 2')
-
Text: حدود دو سال پیش از ترکیه خریدم. عالیه. قسمتی که کودک درونش قرار می گیرد جدا شده و در نوزادی من به عنوان کریر از آن استفاده کردم. اکنون که فرزندم بیست ماهشه ، به شکل نشسته در میاورم و هر وقت خوابید به شکل خوابیده. طوری ساخته شده که وقتی باد بوزد، کودک از باد حفظ است. با کاوری که دارد در باران هم قطره ای آب نفوذ نمی کند. بدنه بسیار مستحکم است. چرخها محکم و به راحتی دور زده میشوند بدون اینکه کالسکه را بلند کنم. صندلی از نظر ارگونامی استاندارد است و به فر

In [4]:
import torch
import time
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import PrefixTuningConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# ۱. بارگذاری دیتاست و توکنایزر
print("Loading Dataset and Tokenizer...")
dataset = load_dataset("hezarai/sentiment-dksf")
train_data = dataset['train']
test_data = dataset['test']

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
label_map = {0: "منفی", 1: "خنثی", 2: "مثبت"}

# ۲. تابع پردازش داده‌ها با ماسک‌گذاری هوشمند پرامپت (Masking)
def preprocess_function_masked(example):
    prompt = f"متن: {example['text']}\nاحساس:"
    label_text = label_map.get(example['label'], "")

    # ساخت پرامپت ورودی کاربر
    user_msg = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=True)

    # متن پاسخ هدف
    target_text = f"{label_text}{tokenizer.eos_token}"

    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    target_ids = tokenizer.encode(target_text, add_special_tokens=False)

    # ترکیب ورودی‌ها
    input_ids = prompt_ids + target_ids
    # قرار دادن 100- برای پرامپت جهت عدم محاسبه خطا روی متن ورودی
    labels = [-100] * len(prompt_ids) + target_ids

    # برش داده‌های طولانی
    if len(input_ids) > 512:
        input_ids = input_ids[-512:]
        labels = labels[-512:]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

print("Tokenizing and Masking data...")
tokenized_train_data = train_data.map(
    preprocess_function_masked,
    remove_columns=train_data.column_names,
    desc="Processing Train Data"
)

# ۳. بارگذاری مدل پایه و تنظیمات Prefix Tuning
print("\nLoading Model for Prefix Tuning...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_pt = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
model_pt = prepare_model_for_kbit_training(model_pt, use_gradient_checkpointing=False)

# افزایش تعداد توکن‌های مجازی به ۳۰ برای یادگیری بهتر
prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=30
)

model_pt = get_peft_model(model_pt, prefix_config)

print("\nPrefix Tuning Trainable Parameters:")
model_pt.print_trainable_parameters()

# ۴. تنظیمات آموزش بهینه‌شده
training_args_pt = TrainingArguments(
    output_dir="./prefix_sentiment_optimized",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-3,             # افزایش نرخ یادگیری مخصوص Prefix Tuning
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none"
)

# استفاده از DataCollator مناسب برای پدینگ لیبل‌ها با 100-
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

trainer_pt = Trainer(
    model=model_pt,
    train_dataset=tokenized_train_data,
    args=training_args_pt,
    data_collator=data_collator
)

# ۵. شروع آموزش و محاسبه زمان
print("\nStarting Optimized Prefix Tuning Training...")
start_pt_train = time.time()

trainer_pt.train()

end_pt_train = time.time()
pt_training_time = end_pt_train - start_pt_train
print(f"\n==========================================")
print(f"Prefix Tuning Training Time: {pt_training_time:.2f} seconds ({pt_training_time/60:.2f} minutes)")
print(f"==========================================")

# ۶. ذخیره آداپتور اصلاح‌شده
trainer_pt.model.save_pretrained("adapter_prefix_optimized")
print("Optimized adapter saved successfully in 'adapter_prefix_optimized' folder.")


# ==========================================
# ۷. ارزیابی مدل بهینه‌شده Prefix Tuning
# ==========================================

def clean_and_extract_label(gen_text):
    gen_text = gen_text.strip()
    if "مثبت" in gen_text or gen_text.startswith("مث"):
        return "مثبت"
    elif "منفی" in gen_text or gen_text.startswith("من"):
        return "منفی"
    elif "خنثی" in gen_text or gen_text.startswith("خ"):
        return "خنثی"
    return gen_text

def evaluate_prefix_model(model, tokenizer, test_dataset, num_samples=500):
    test_subset = test_dataset.select(range(min(num_samples, len(test_dataset))))

    true_labels = []
    pred_labels = []
    errors = []

    model.eval()
    for row in tqdm(test_subset, desc="Evaluating"):
        prompt = f"متن: {row['text']}\nاحساس:"
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        gen_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = clean_and_extract_label(gen_text)
        true_label = label_map.get(row['label'], "")

        true_labels.append(true_label)
        pred_labels.append(pred)

        if pred != true_label and len(errors) < 5:
            errors.append({"text": row['text'], "true": true_label, "pred": pred, "raw": gen_text})

    acc = accuracy_score(true_labels, pred_labels)
    labels_list = ["مثبت", "منفی", "خنثی"]
    f_macro = f1_score(true_labels, pred_labels, labels=labels_list, average='macro', zero_division=0)

    return acc, f_macro, errors

print("\nEvaluating Optimized Prefix Tuning Model...")
acc_pt, f_macro_pt, error_samples_pt = evaluate_prefix_model(trainer_pt.model, tokenizer, test_data)

print(f"\n==========================================")
print(f"Optimized Prefix Tuning Accuracy : {acc_pt:.4f} ({acc_pt*100:.2f}%)")
print(f"Optimized Prefix Tuning F-Macro  : {f_macro_pt:.4f}")
print(f"==========================================")

print("\n--- Error Samples (Optimized Prefix Tuning) ---")
for e in error_samples_pt:
    print(f"Text: {e['text']}\nTrue: {e['true']} | Predicted: {e['pred']} (Raw: '{e['raw']}')\n-")

Loading Dataset and Tokenizer...
Tokenizing and Masking data...


Processing Train Data:   0%|          | 0/28602 [00:00<?, ? examples/s]


Loading Model for Prefix Tuning...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Prefix Tuning Trainable Parameters:
trainable params: 184,320 || all params: 494,217,088 || trainable%: 0.0373

Starting Optimized Prefix Tuning Training...


Step,Training Loss
50,11.634110
100,4.888058
150,1.331167
200,0.510786
250,0.379337
300,0.337982
350,0.336870
400,0.320464
450,0.327852
500,0.319093



Prefix Tuning Training Time: 1735.32 seconds (28.92 minutes)
Optimized adapter saved successfully in 'adapter_prefix_optimized' folder.

Evaluating Optimized Prefix Tuning Model...


Evaluating: 100%|██████████| 500/500 [02:25<00:00,  3.42it/s]


Optimized Prefix Tuning Accuracy : 0.6720 (67.20%)
Optimized Prefix Tuning F-Macro  : 0.4820

--- Error Samples (Optimized Prefix Tuning) ---
Text: قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا
True: خنثی | Predicted: منفی (Raw: 'منفی')
-
Text: حدود دو سال پیش از ترکیه خریدم. عالیه. قسمتی که کودک درونش قرار می گیرد جدا شده و در نوزادی من به عنوان کریر از آن استفاده کردم. اکنون که فرزندم بیست ماهشه ، به شکل نشسته در میاورم و هر وقت خوابید به شکل خوابیده. طوری ساخته شده که وقتی باد بوزد، کودک از باد حفظ است. با کاوری که دارد در باران هم قطره ای آب نفوذ نمی کند. بدنه بسیار مستحکم است. چرخها محکم و به راحتی دور زده میشوند بدون اینکه کالسکه را بلند کنم. صندلی از نظر ارگونامی استاندارد است و به فرم کمر و بدن کودک آسیب نمی زند. بسیار استاندارد و عالی است.
True: خنثی | Predicted: منفی (Raw: 'منفی')
-
Text: خیلی خیلی سرد بود و سیب زمینی با قارچ کیفیت همیشگی رو نداشت.
True: منفی | Predicted: خنثی (Raw: 'خنثی')
-
Text: مقدار حجم ساندویج و کیفیتش به نسبت اسم هایدا خیلی خیلی پایین بود البته سالهاست که